In [5]:
import os

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    temperature=0.0,
    api_key=os.getenv("DEEPSEEK_API"),
    model_provider="openai",
)

from langchain.tools import tool


@tool
def dangerous_write(sql: str) -> str:
    """对数据库执行操作，当请求涉及数据库操作活存在'SQL:'字样时，使用本工具

    Args:
        sql (str): sql语句

    Returns:
        str: 结果
    """
    print(f"TOOL: 执行SQL: {sql}")
    return f"模拟执行SQL: {sql}"


from langchain.agents.middleware import HumanInTheLoopMiddleware

hitl = HumanInTheLoopMiddleware(interrupt_on={"dangerous_write": {"allowed_decisions": ["approve", "edit", "reject"]}})

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    tools=[dangerous_write],
    middleware=[hitl],
    system_prompt="""
        只用一句简短的中文回答, 不超过20字;
        凡是涉及数据库操作, 或者消息以'SQL:'开头，使用工具 dangerous_write, 不得直接回答
        """,
    checkpointer=InMemorySaver(),
)

CFG = {"configurable": {"thread_id": "hitl-demo-interactive"}}

conversation = [
    "你是谁?",
    "SQL: INSERT INTO logs(content) VALUES ('hello')",
    "继续",
    "SQL: DELETE FROM orders WHERE id = '123'",
]

state = {"messages": []}
from langgraph.types import Command


def handle_interrupt(result) -> dict:
    interrupt = result.get("__interrupt__")
    if not interrupt:
        return result
    decision = input("请输入决策\n1: approve\n2: edit\n3: reject\n:")
    if decision == "1":
        return agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=CFG)
    elif decision == "2":
        new_sql = input("请输入修改后的 SQL：").strip()
        return agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {"type": "edit", "edited_action": {"name": "dangerous_write", "args": {"sql": new_sql}}}
                    ]
                }
            ),
            config=CFG,
        )
    else:
        return agent.invoke(
            Command(resume={"decisions": [{"type": "reject", "override": {"contnet": ["操作已被人工拒绝"]}}]}),
            config=CFG,
        )


from langchain.messages import HumanMessage

for i, q in enumerate(conversation, 1):
    print(f"第{i}论会话: ")
    print(f"Q: {q}")
    user_msg = HumanMessage(content=q)
    result = await agent.ainvoke({"messages": state["messages"] + [user_msg]}, config=CFG)
    if result.get("__interrupt__"):
        result = handle_interrupt(result=result)
    state = result
    ans = state["messages"][-1].content
    print(f"A: {ans}")
print("历史消息")
for i in state["messages"]:
    print(f"{type(i).__name__}: {i}")

第1论会话: 
Q: 你是谁?
A: 我是AI助手，很高兴为你服务！
第2论会话: 
Q: SQL: INSERT INTO logs(content) VALUES ('hello')
TOOL: 执行SQL: INSERT INTO logs(content) VALUES ('hello')
A: 已执行数据库操作。
第3论会话: 
Q: 继续
TOOL: 执行SQL: yangxb
A: 已执行。
第4论会话: 
Q: SQL: DELETE FROM orders WHERE id = '123'
A: 操作已被拒绝，无法执行删除。
历史消息
HumanMessage: content='你是谁?' additional_kwargs={} response_metadata={} id='f24a601b-0908-452b-941e-69b9b9e3fce0'
AIMessage: content='我是AI助手，很高兴为你服务！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 348, 'total_tokens': 356, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 92}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '528779ba-1cdd-42e9-b3bf-d5feac0cadb0', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f2243-a9de-7ae3-a702-c878